# ULTRON owner setup — preflight only

**This notebook does not train.** It does not instantiate the model, create an optimizer, generate or download a dataset, perform optimizer steps, or dispatch remote jobs.

Purpose: confirm the repository is present, load the website-exported schema-v1 runtime configuration, and independently calculate the foundation model's parameter counts from configuration formulas. The notebook now verifies an existing dataset and constructs read-only split DataLoaders; checkpoint resume and training remain disabled.

The config file is untrusted input. Its authorization flags must remain false.

In [ ]:
from pathlib import Path
import json
import platform
import sys

repo_root = Path.cwd()
if not (repo_root / "ultron_core").is_dir():
    raise FileNotFoundError("Run this notebook from the ULTRON repository root (ultron_core/ must exist).")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print({"python": platform.python_version(), "platform": platform.platform(), "repo_root": str(repo_root)})

## Load the website-exported config

On the device/browser, export `ultron-runtime-config.json` from ULTRON Settings, then upload it to this notebook runtime using its file-upload UI. Enter the runtime-local path below. This is a manual file handoff; the website does not connect to this notebook.

In [ ]:
from ultron_core.runtime_config import RuntimeConfig, MAX_CONFIG_JSON_BYTES

config_path = Path(input("Runtime-local path to ultron-runtime-config.json: ").strip()).expanduser()
if not config_path.is_file():
    raise FileNotFoundError(f"Config file not found: {config_path}")
config_bytes = config_path.read_bytes()
if len(config_bytes) > MAX_CONFIG_JSON_BYTES:
    raise ValueError("Config exceeds the schema's 16 KiB limit.")
runtime_config = RuntimeConfig.from_json(config_bytes.decode("utf-8"))
assert runtime_config.training_authorized is False
assert runtime_config.remote_execution_enabled is False
print(json.dumps(runtime_config.to_dict(), indent=2, sort_keys=True))

## Capacity formula check — no model allocation

This constructs only the small immutable configuration object. The count functions use parameter-shape formulas; they do not instantiate `NeuroSparseModel` or allocate its weights.

In [ ]:
from ultron_core.config import NeuroSparseConfig, count_parameters, count_total_parameters, MIN_FOUNDATION_TRAINABLE_PARAMETERS

foundation_config = NeuroSparseConfig.foundation()
trainable_count = count_parameters(foundation_config)
total_count = count_total_parameters(foundation_config)
assert trainable_count >= MIN_FOUNDATION_TRAINABLE_PARAMETERS
assert total_count >= trainable_count
assert foundation_config.vocab_size == 240
print({
    "total_parameters_by_formula": total_count,
    "trainable_parameters_by_formula": trainable_count,
    "minimum_trainable_parameters": MIN_FOUNDATION_TRAINABLE_PARAMETERS,
    "sequence_length_limit": foundation_config.max_sequence_length,
    "runtime_budget_gib_requested": runtime_config.memory_budget_gib,
    "batch_size_requested": runtime_config.batch_size,
    "sequence_length_requested": runtime_config.sequence_length,
    "gradient_accumulation_requested": runtime_config.gradient_accumulation_steps,
})

## Verify an existing dataset and prepare split metadata

Upload an **already-existing** ULTRON JSONL dataset and its `.manifest.json` sidecar. This verifies the dataset and writes only deterministic split-ID metadata. It does not create or edit dataset rows, download data, or train.

In [ ]:
from ultron_core.dataset_split_manifest import prepare_dataset_split_manifest

existing_dataset_path = Path(input("Path to existing verified ULTRON JSONL dataset: ").strip()).expanduser()
split_manifest_path = Path(input("Output path for split-ID metadata JSON: ").strip()).expanduser()
split_seed_text = input("Deterministic split seed [0]: ").strip()
split_seed = int(split_seed_text) if split_seed_text else 0
# prepare_dataset_split_manifest verifies the result before returning.
split_contract = prepare_dataset_split_manifest(
    existing_dataset_path,
    split_manifest_path,
    split_seed=split_seed,
    validation_fraction=0.1,
    test_fraction=0.1,
)
print({
    "dataset_sha256": split_contract.dataset_sha256,
    "split_manifest_sha256": split_contract.split_manifest_sha256,
    "dataset_examples": split_contract.dataset_example_count,
    "selected_unique_prompt_examples": split_contract.selected_example_count,
    "split_counts": split_contract.split_counts,
    "status": "DATASET_AND_SPLITS_VERIFIED / PREFLIGHT_ONLY",
})

## Build verified loaders and a complete non-authorizing preflight report

This stage reuses one verified split snapshot to build the three read-only DataLoaders and validate every selected prompt/answer through the canonical tokenizer. It reports exact parameter counts, persistent-state memory estimates, dataset/split digests, split counts, and token-length limits.

The sampler seed is deliberately separate from the split seed. The reported sampler seed must later match `TrainingSettings.seed`. No batches are requested from any DataLoader.

In [ ]:
from ultron_core.verified_preflight import build_verified_training_preflight

# Keep this in sync with TrainingSettings.seed if a separately authorized run
# is ever configured. It is not the prompt-group split seed above.
sampler_seed = 0
verified_preflight = build_verified_training_preflight(
    existing_dataset_path,
    split_manifest_path,
    runtime_config,
    model_config=foundation_config,
    sampler_seed=sampler_seed,
)
preflight_report = verified_preflight.report
train_loader = verified_preflight.train_loader
validation_loader = verified_preflight.validation_loader
test_loader = verified_preflight.test_loader
initial_data_state = verified_preflight.initial_data_state

print(json.dumps(preflight_report, indent=2, sort_keys=True))
print({
    "train_examples": len(train_loader.dataset),
    "validation_examples": len(validation_loader.dataset),
    "test_examples": len(test_loader.dataset),
    "sampler_seed": sampler_seed,
    "initial_checkpoint_data_state": initial_data_state,
    "loader_contract": {
        "num_workers": train_loader.num_workers,
        "drop_last": train_loader.drop_last,
        "evaluation_samplers": [
            type(validation_loader.sampler).__name__,
            type(test_loader.sampler).__name__,
        ],
    },
    "status": "VERIFIED_PREFLIGHT_REPORT_READY / NO_BATCHES_CONSUMED / NO_TRAINING",
})

## Stop point

Expected result: `CONFIG_VALIDATED / CAPACITY_FORMULA_CHECKED / DATASET_AND_SPLITS_VERIFIED / TOKENIZATION_VALIDATED / VERIFIED_PREFLIGHT_REPORT_READY / NO_BATCHES_CONSUMED`.

The report is read-only and explicitly non-authorizing. The three DataLoaders are built from one verified split snapshot. No batch is consumed; no model weights or optimizer are allocated; no checkpoint is restored; no optimizer step, training, dataset generation/download, or remote job is performed.

The persistent-state memory estimate covers FP32 weights, gradients, and AdamW moments only. It does **not** claim a complete device-memory estimate: dataset Python-object residency, activations, attention workspaces, temporary tensors, runtime overhead, and allocator fragmentation remain unmeasured. The sampler seed in `initial_data_state` must match `TrainingSettings.seed`.

Next engineering stage: connect the initial cursor and sampler state to the existing checkpoint callback and test metadata-only save/load contracts. Actual checkpoint restoration, batch consumption, optimizer steps, and uninterrupted-versus-resumed equivalence remain separately gated.